# Price Data Preprocessing Pipeline

**Purpose:** Standardize all DA LMP/SPP datasets into two unified Parquet files:

1. **`hourly_zone_prices.parquet`** — All hourly-resolution zones (PJM, ERCOT, CAISO, MISO, etc.)
2. **`daily_peak_zone_prices.parquet`** — Daily on-peak averages for ALL zones (hourly collapsed + native daily Mid-C/Palo Verde)

Plus a metadata file:
3. **`zone_metadata.parquet`** — Zone ID, RTO, interconnection, DC capacity (MW), migration role

**Common schema:** `datetime, price, zone_id, rto, interconnection`

Run this once after acquiring new data. Analysis notebooks load the Parquet files only.

In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── Repo root (resolved from this notebook's location) ───────────────────────
REPO_ROOT = Path(__file__).resolve().parent.parent if '__file__' in dir() else Path.cwd().parent
if not (REPO_ROOT / 'notebooks').exists():
    REPO_ROOT = Path.cwd()
    while not (REPO_ROOT / 'notebooks').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / 'notebooks').exists(), (
    f"Could not find repo root. CWD={Path.cwd()}, tried={REPO_ROOT}"
)

# ── Raw price data (large files, stay in OneDrive, referenced via env var) ───
PRICE_DIR = os.environ.get('THESIS_PRICE_DIR')
if PRICE_DIR is None:
    PRICE_DIR = r'C:\Users\dunla\OneDrive\Documents\Bartlett Fellowship\Thesis\Data\Power_Prices'
    print(f"WARNING: THESIS_PRICE_DIR env var not set, using fallback: {PRICE_DIR}")
assert os.path.isdir(PRICE_DIR), f"Price data directory not found: {PRICE_DIR}"

# ── DC capacity source data (now in repo) ────────────────────────────────────
DC_CAPACITY_CSV = REPO_ROOT / 'data' / 'raw' / 'dc_capacity_mapped.csv'

# ── Output directory (processed parquets go into repo) ───────────────────────
OUTPUT_DIR = REPO_ROOT / 'data' / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS = [2022, 2023, 2024, 2025]

# ── On-peak hour definition ──────────────────────────────────────────────────
PEAK_HOURS_START = 6
PEAK_HOURS_END   = 21

print(f'Repo root:           {REPO_ROOT}')
print(f'Price data (raw):    {PRICE_DIR}')
print(f'DC capacity CSV:     {DC_CAPACITY_CSV}')
print(f'Output (processed):  {OUTPUT_DIR}')
print(f'Peak hours: HB {PEAK_HOURS_START}:00 – {PEAK_HOURS_END}:00 (HE{PEAK_HOURS_START+1}–HE{PEAK_HOURS_END+1})')
print(f'Years: {YEARS}')
print(f'\nAvailable price files:')
for f in sorted(os.listdir(PRICE_DIR)):
    size = os.path.getsize(os.path.join(PRICE_DIR, f)) / 1024
    print(f'  {f:<50} {size:>8,.0f} KB')

Repo root:           C:\Users\dunla\repos\data-center-flexibility-resource-adequacy
Price data (raw):    C:\Users\dunla\OneDrive\Documents\Bartlett Fellowship\Thesis\Data\Power_Prices
DC capacity CSV:     C:\Users\dunla\repos\data-center-flexibility-resource-adequacy\data\raw\dc_capacity_mapped.csv
Output (processed):  C:\Users\dunla\repos\data-center-flexibility-resource-adequacy\data\processed
Peak hours: HB 6:00 – 21:00 (HE7–HE22)
Years: [2022, 2023, 2024, 2025]

Available price files:
  OASIS_Day-Ahead_Market_Zonal_LBMP.csv                     0 KB
  Palo_Verde_Mid_C_2022.xlsx                               94 KB
  Palo_Verde_Mid_C_2023.xlsx                               98 KB
  Palo_Verde_Mid_C_2024.xlsx                               88 KB
  Palo_Verde_Mid_C_2025.xlsx                               90 KB
  caiso_np15_da_lmp_2022_2025.csv                       4,118 KB
  caiso_sp15_da_lmp_2022_2025.csv                       4,141 KB
  ercot_LZ_HOUSTON_da_spp_2022_2025.csv            

In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# ZONE METADATA — from Baxtel/DOE county mapping
# ══════════════════════════════════════════════════════════════════════════════
# zone_id must match exactly what we assign in the loader functions below

ZONE_META = pd.DataFrame([
    # ── PJM (source + intra-RTO control) ──────────────────────────────────────
    {'zone_id': 'PJM_COMED',   'rto': 'PJM',   'interconnection': 'Eastern', 'dc_capacity_mw': 1644.5,  'migration_role': 'source'},
    {'zone_id': 'PJM_DOM',     'rto': 'PJM',   'interconnection': 'Eastern', 'dc_capacity_mw': 9430.7,  'migration_role': 'intra_rto_control'},
    {'zone_id': 'PJM_AEP',     'rto': 'PJM',   'interconnection': 'Eastern', 'dc_capacity_mw': 3506.6,  'migration_role': 'intra_rto_control'},
    {'zone_id': 'PJM_BGE',     'rto': 'PJM',   'interconnection': 'Eastern', 'dc_capacity_mw': 74.7,    'migration_role': 'intra_rto_control'},
    {'zone_id': 'PJM_PECO',    'rto': 'PJM',   'interconnection': 'Eastern', 'dc_capacity_mw': 694.5,   'migration_role': 'intra_rto_control'},
    {'zone_id': 'PJM_PEPCO',   'rto': 'PJM',   'interconnection': 'Eastern', 'dc_capacity_mw': 10.0,   'migration_role': 'intra_rto_control'},
    {'zone_id': 'PJM_PSEG',    'rto': 'PJM',   'interconnection': 'Eastern', 'dc_capacity_mw': 747.2,   'migration_role': 'intra_rto_control'},
    
    # ── ERCOT (cross-IC destination) ──────────────────────────────────────────
    {'zone_id': 'ERCOT_LZ_NORTH',   'rto': 'ERCOT', 'interconnection': 'ERCOT',  'dc_capacity_mw': 4274.6,  'migration_role': 'cross_ba_destination'},
    {'zone_id': 'ERCOT_LZ_SOUTH',   'rto': 'ERCOT', 'interconnection': 'ERCOT',  'dc_capacity_mw': 2866.1,  'migration_role': 'cross_ba_destination'},
    {'zone_id': 'ERCOT_LZ_WEST',    'rto': 'ERCOT', 'interconnection': 'ERCOT',  'dc_capacity_mw': 2480.0,  'migration_role': 'cross_ba_destination'},
    {'zone_id': 'ERCOT_LZ_HOUSTON', 'rto': 'ERCOT', 'interconnection': 'ERCOT',  'dc_capacity_mw': 889.8,   'migration_role': 'cross_ba_destination'},
    
    # ── CAISO (cross-IC destination) ──────────────────────────────────────────
    {'zone_id': 'CAISO_NP15',  'rto': 'CAISO', 'interconnection': 'Western', 'dc_capacity_mw': 1310.7,  'migration_role': 'cross_ba_destination'},
    {'zone_id': 'CAISO_SP15',  'rto': 'CAISO', 'interconnection': 'Western', 'dc_capacity_mw': 396.8,   'migration_role': 'cross_ba_destination'},
    
    # ── WECC bilateral hubs (cross-IC destination, daily resolution) ──────────
    {'zone_id': 'WECC_MIDC',       'rto': 'WECC-nonISO', 'interconnection': 'Western', 'dc_capacity_mw': 5554.2, 'migration_role': 'cross_ba_destination'},
    {'zone_id': 'WECC_PALO_VERDE', 'rto': 'WECC-nonISO', 'interconnection': 'Western', 'dc_capacity_mw': 4508.7, 'migration_role': 'cross_ba_destination'},
    
    # ── MISO (cross-BA destination — same Eastern IC but independent RA) ─────
    {'zone_id': 'MISO_MINN_HUB',      'rto': 'MISO', 'interconnection': 'Eastern', 'dc_capacity_mw': 2988.3, 'migration_role': 'cross_ba_destination'},
    {'zone_id': 'MISO_INDIANA_HUB',   'rto': 'MISO', 'interconnection': 'Eastern', 'dc_capacity_mw': 598.6,  'migration_role': 'cross_ba_destination'},
    {'zone_id': 'MISO_ILLINOIS_HUB',  'rto': 'MISO', 'interconnection': 'Eastern', 'dc_capacity_mw': 66.2,   'migration_role': 'cross_ba_destination'},
    {'zone_id': 'MISO_MICHIGAN_HUB',  'rto': 'MISO', 'interconnection': 'Eastern', 'dc_capacity_mw': 126.3,  'migration_role': 'cross_ba_destination'},
    {'zone_id': 'MISO_ARKANSAS_HUB',  'rto': 'MISO', 'interconnection': 'Eastern', 'dc_capacity_mw': 305.0,  'migration_role': 'cross_ba_destination'},
    {'zone_id': 'MISO_LOUISIANA_HUB', 'rto': 'MISO', 'interconnection': 'Eastern', 'dc_capacity_mw': 7.0,    'migration_role': 'cross_ba_destination'},
    
    # ── NYISO (cross-BA destination — same Eastern IC) ──────────────────────
    {'zone_id': 'NYISO_ZONE_J',  'rto': 'NYISO',  'interconnection': 'Eastern', 'dc_capacity_mw': 191.3,  'migration_role': 'cross_ba_destination'},
    {'zone_id': 'NYISO_ZONE_F',  'rto': 'NYISO',  'interconnection': 'Eastern', 'dc_capacity_mw': 548.5,  'migration_role': 'cross_ba_destination'},
    {'zone_id': 'NYISO_ZONE_A',  'rto': 'NYISO',  'interconnection': 'Eastern', 'dc_capacity_mw': 55.6,   'migration_role': 'cross_ba_destination'},
    {'zone_id': 'NYISO_ZONE_G',  'rto': 'NYISO',  'interconnection': 'Eastern', 'dc_capacity_mw': 86.1,   'migration_role': 'cross_ba_destination'},
    {'zone_id': 'NYISO_ZONE_K',  'rto': 'NYISO',  'interconnection': 'Eastern', 'dc_capacity_mw': 9.5,    'migration_role': 'cross_ba_destination'},
    # {'zone_id': 'ISONE_HUB',     'rto': 'ISO-NE', 'interconnection': 'Eastern', 'dc_capacity_mw': 208.2,  'migration_role': 'cross_ba_destination'},
    # {'zone_id': 'SPP_HUB',       'rto': 'SPP',    'interconnection': 'Eastern', 'dc_capacity_mw': 1824.7, 'migration_role': 'cross_ba_destination'},
])

ZONE_META = ZONE_META.set_index('zone_id')

print(f'Zone metadata: {len(ZONE_META)} zones')
print(f'\nBy migration role:')
for role, grp in ZONE_META.groupby('migration_role'):
    print(f'  {role:<25} {len(grp):>3} zones  {grp.dc_capacity_mw.sum():>10,.1f} MW')
print(f'\nBy interconnection:')
for ic, grp in ZONE_META.groupby('interconnection'):
    print(f'  {ic:<25} {len(grp):>3} zones  {grp.dc_capacity_mw.sum():>10,.1f} MW')

Zone metadata: 26 zones

By migration role:
  cross_ba_destination       19 zones    27,263.3 MW
  intra_rto_control           6 zones    14,463.7 MW
  source                      1 zones     1,644.5 MW

By interconnection:
  ERCOT                       4 zones    10,510.5 MW
  Eastern                    18 zones    21,090.6 MW
  Western                     4 zones    11,770.4 MW


In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# LOADER FUNCTIONS — one per source format
# ══════════════════════════════════════════════════════════════════════════════

def load_pjm_zone(filepath, zone_id):
    """
    PJM Data Miner format.
    Columns: datetime_beginning_ept, total_lmp_da, pnode_name, version_nbr, row_is_current
    """
    df = pd.read_csv(filepath)
    
    # Extract zone name from zone_id (PJM_COMED → COMED)
    pnode = zone_id.replace('PJM_', '')
    
    # Filter to correct pnode if file has multiple
    if 'pnode_name' in df.columns:
        df = df[df['pnode_name'].str.upper() == pnode].copy()
    
    df['datetime'] = pd.to_datetime(df['datetime_beginning_ept'])
    df['price'] = pd.to_numeric(df['total_lmp_da'], errors='coerce')
    
    # De-duplicate: keep latest version
    sort_cols = ['datetime']
    if 'row_is_current' in df.columns:
        sort_cols.append('row_is_current')
    if 'version_nbr' in df.columns:
        sort_cols.append('version_nbr')
    df = df.sort_values(sort_cols).drop_duplicates(subset=['datetime'], keep='last')
    
    df['year'] = df['datetime'].dt.year
    df = df[df['year'].isin(YEARS)]
    
    return df[['datetime', 'price']].set_index('datetime').sort_index()


def load_ercot_zone(filepath, zone_id):
    """
    ERCOT gridstatus format.
    Columns vary — typically: Interval Start, SPP (or similar price col)
    May have Location column if multi-zone file, or be single-zone.
    Times are UTC → convert to EPT for alignment.
    """
    df = pd.read_csv(filepath)
    
    # Find datetime column
    dt_col = None
    for candidate in ['Interval Start', 'Time', 'datetime', 'Datetime']:
        if candidate in df.columns:
            dt_col = candidate
            break
    if dt_col is None:
        raise ValueError(f'No datetime column found. Columns: {df.columns.tolist()}')
    
    # Find price column
    price_col = None
    for candidate in ['SPP', 'LMP', 'Price', 'price']:
        if candidate in df.columns:
            price_col = candidate
            break
    if price_col is None:
        raise ValueError(f'No price column found. Columns: {df.columns.tolist()}')
    
    df['datetime'] = pd.to_datetime(df[dt_col], utc=True)
    df['datetime'] = df['datetime'].dt.tz_convert('US/Eastern').dt.tz_localize(None)
    df['price'] = pd.to_numeric(df[price_col], errors='coerce')
    
    df['year'] = df['datetime'].dt.year
    df = df[df['year'].isin(YEARS)]
    df = df.drop_duplicates(subset=['datetime'], keep='last')
    
    return df[['datetime', 'price']].set_index('datetime').sort_index()


def load_caiso_zone(filepath, zone_id):
    """
    CAISO format — similar to ERCOT gridstatus.
    May start Nov 2022 (not full year). That's fine.
    """
    # Same logic as ERCOT — gridstatus produces similar format
    return load_ercot_zone(filepath, zone_id)


def load_miso_hub(filepath, hub_node_name, zone_id):
    """
    MISO scraper output format.
    Columns: date, hour_ending, datetime, price, node
    File may contain multiple hubs — filter by hub_node_name.
    """
    df = pd.read_csv(filepath)
    
    # Filter to specific hub if multi-hub file
    if 'node' in df.columns:
        df = df[df['node'].str.upper() == hub_node_name.upper()].copy()
    
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['price'] = pd.to_numeric(df['price'], errors='coerce')
    
    df['year'] = df['datetime'].dt.year
    df = df[df['year'].isin(YEARS)]
    df = df.drop_duplicates(subset=['datetime'], keep='last')
    
    return df[['datetime', 'price']].set_index('datetime').sort_index()


def load_eia_daily(filepath, hub_name, zone_id):
    """
    EIA/ICE daily wholesale price format (Mid-C, Palo Verde).
    Columns: Price hub, Trade date, Delivery start date, Delivery end date,
             High price $/MWh, Low price $/MWh, Wtd avg price $/MWh, ...
    
    Returns daily-resolution DataFrame (not hourly).
    Uses Wtd avg price as the price signal.
    """
    df = pd.read_excel(filepath)
    
    # Filter to correct hub if file has both
    if 'Price hub' in df.columns:
        df = df[df['Price hub'].str.contains(hub_name, case=False, na=False)].copy()
    
    # Find date column
    date_col = None
    for candidate in ['Delivery start date', 'Delivery start d', 'Trade date']:
        matches = [c for c in df.columns if candidate.lower() in c.lower()]
        if matches:
            date_col = matches[0]
            break
    if date_col is None:
        # Try first date-like column
        for c in df.columns:
            if 'date' in c.lower() or 'delivery' in c.lower():
                date_col = c
                break
    if date_col is None:
        raise ValueError(f'No date column found. Columns: {df.columns.tolist()}')
    
    # Find price column
    price_col = None
    for candidate in ['Wtd avg price', 'Wtd avg']:
        matches = [c for c in df.columns if candidate.lower() in c.lower()]
        if matches:
            price_col = matches[0]
            break
    if price_col is None:
        raise ValueError(f'No wtd avg price column found. Columns: {df.columns.tolist()}')
    
    df['date'] = pd.to_datetime(df[date_col])
    df['price'] = pd.to_numeric(df[price_col], errors='coerce')
    
    df['year'] = df['date'].dt.year
    df = df[df['year'].isin(YEARS)]
    df = df.drop_duplicates(subset=['date'], keep='last')
    
    return df[['date', 'price']].set_index('date').sort_index()


# ── NYISO loader (GridStatus format) ─────────────────────────────────────────

def load_nyiso_zone(filepath, zone_id):
    """
    NYISO GridStatus format.
    Columns: Time, Interval Start, Interval End, Market, Location,
             Location Type, LMP, Energy, Congestion, Loss
    Timestamps are timezone-aware (US/Eastern, -05:00).
    Each file is a single zone, so Location column is constant — but we
    validate it against the expected zone name for safety.
    """
    df = pd.read_csv(filepath)
    
    zone_name = zone_id.replace('NYISO_', '')  # e.g., ZONE_J
    # NYISO GridStatus Location values per zone letter
    NYISO_ZONE_MAP = {
        'ZONE_J': 'N.Y.C.', 'ZONE_F': 'CAPITL', 'ZONE_G': 'HUD VL',
        'ZONE_A': 'WEST', 'ZONE_K': 'LONGIL',
    }
    expected_loc = NYISO_ZONE_MAP.get(zone_name, zone_name)
    
    # Validate location if column present (GridStatus single-zone files)
    if 'Location' in df.columns:
        actual_locs = df['Location'].dropna().unique()
        if len(actual_locs) == 1 and actual_locs[0].strip() != expected_loc:
            warnings.warn(f'{zone_id}: expected Location="{expected_loc}", got "{actual_locs[0]}"')
    
    # Find datetime column — GridStatus uses "Interval Start"
    dt_col = None
    for candidate in ['Interval Start', 'Time', 'datetime']:
        if candidate in df.columns:
            dt_col = candidate
            break
    if dt_col is None:
        raise ValueError(f'No datetime column found. Columns: {df.columns.tolist()}')
    
    # Parse timezone-aware timestamps → convert to naive Eastern for alignment
    # NYISO GridStatus uses Eastern Prevailing Time (mixed -05:00/-04:00 offsets)
    # Parse as UTC first to avoid mixed-tz warning, then convert to Eastern naive
    df['datetime'] = pd.to_datetime(df[dt_col], utc=True)
    df['datetime'] = df['datetime'].dt.tz_convert('US/Eastern').dt.tz_localize(None)
    
    df['price'] = pd.to_numeric(df['LMP'], errors='coerce')
    
    df['year'] = df['datetime'].dt.year
    df = df[df['year'].isin(YEARS)]
    df = df.drop_duplicates(subset=['datetime'], keep='last')
    
    return df[['datetime', 'price']].set_index('datetime').sort_index()


# def load_isone_hub(filepath, zone_id):
#     """
#     ISO-NE DA LMP.
#     Download from: https://www.iso-ne.com/isoexpress/web/reports/pricing
#     CSV format: Date, Hour Ending, DA_LMP
#     """
#     df = pd.read_csv(filepath)
#     df['datetime'] = pd.to_datetime(df['Date']) + pd.to_timedelta(df['Hour Ending'] - 1, unit='h')
#     df['price'] = pd.to_numeric(df['DA_LMP'], errors='coerce')
#     df['year'] = df['datetime'].dt.year
#     df = df[df['year'].isin(YEARS)]
#     df = df.drop_duplicates(subset=['datetime'], keep='last')
#     return df[['datetime', 'price']].set_index('datetime').sort_index()


# def load_spp_hub(filepath, zone_id):
#     """
#     SPP DA LMP.
#     Download from: https://marketplace.spp.org/pages/da-lmp-by-location
#     CSV format varies — adapt columns after first download.
#     """
#     df = pd.read_csv(filepath)
#     # Adapt column names after inspecting first download
#     df['datetime'] = pd.to_datetime(df['GMTIntervalEnd'], utc=True)
#     df['datetime'] = df['datetime'].dt.tz_convert('US/Eastern').dt.tz_localize(None)
#     df['price'] = pd.to_numeric(df['LMP'], errors='coerce')
#     df['year'] = df['datetime'].dt.year
#     df = df[df['year'].isin(YEARS)]
#     df = df.drop_duplicates(subset=['datetime'], keep='last')
#     return df[['datetime', 'price']].set_index('datetime').sort_index()


print('Loader functions defined.')

Loader functions defined.


In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# FILE REGISTRY — maps zone_id → file path + loader function
# ══════════════════════════════════════════════════════════════════════════════
# To add a new zone: add its metadata above, add its file entry here, done.

P = PRICE_DIR  # shorthand

HOURLY_SOURCES = {
    # ── PJM zones ────────────────────────────────────────────────────────────
    'PJM_COMED': {'file': f'{P}/pjm_comed_da_lmp_2022_2025.csv',   'loader': load_pjm_zone},
    'PJM_DOM':   {'file': f'{P}/pjm_dom_da_lmp_2022_2025.csv',     'loader': load_pjm_zone},
    'PJM_AEP':   {'file': f'{P}/pjm_aep_da_lmp_2022_2025.csv',     'loader': load_pjm_zone},
    'PJM_BGE':   {'file': f'{P}/pjm_bge_da_lmp_2022_2025.csv',     'loader': load_pjm_zone},
    'PJM_PECO':  {'file': f'{P}/pjm_peco_da_lmp_2022_2025.csv',    'loader': load_pjm_zone},
    'PJM_PSEG':  {'file': f'{P}/pjm_pseg_da_lmp_2022_2025.csv',    'loader': load_pjm_zone},
    'PJM_PEPCO': {'file': f'{P}/pjm_pepco_da_lmp_2022_2025.csv',   'loader': load_pjm_zone},
    
    # ── ERCOT zones ──────────────────────────────────────────────────────────
    'ERCOT_LZ_NORTH':   {'file': f'{P}/ercot_LZ_NORTH_da_spp_2022_2025.csv',   'loader': load_ercot_zone},
    'ERCOT_LZ_SOUTH':   {'file': f'{P}/ercot_LZ_SOUTH_da_spp_2022_2025.csv',   'loader': load_ercot_zone},
    'ERCOT_LZ_WEST':    {'file': f'{P}/ercot_LZ_WEST_da_spp_2022_2025.csv',    'loader': load_ercot_zone},
    'ERCOT_LZ_HOUSTON': {'file': f'{P}/ercot_LZ_HOUSTON_da_spp_2022_2025.csv', 'loader': load_ercot_zone},
    
    # ── CAISO zones ──────────────────────────────────────────────────────────
    'CAISO_NP15': {'file': f'{P}/caiso_np15_da_lmp_2022_2025.csv', 'loader': load_caiso_zone},
    'CAISO_SP15': {'file': f'{P}/caiso_sp15_da_lmp_2022_2025.csv', 'loader': load_caiso_zone},
    
    # ── MISO hubs ────────────────────────────────────────────────────────────
    'MISO_MINN_HUB':      {'file': f'{P}/miso_all_hubs_da_lmp_2022_2025.csv', 'loader': lambda f, z: load_miso_hub(f, 'MINN.HUB', z)},
    'MISO_INDIANA_HUB':   {'file': f'{P}/miso_all_hubs_da_lmp_2022_2025.csv', 'loader': lambda f, z: load_miso_hub(f, 'INDIANA.HUB', z)},
    'MISO_ILLINOIS_HUB':  {'file': f'{P}/miso_all_hubs_da_lmp_2022_2025.csv', 'loader': lambda f, z: load_miso_hub(f, 'ILLINOIS.HUB', z)},
    'MISO_MICHIGAN_HUB':  {'file': f'{P}/miso_all_hubs_da_lmp_2022_2025.csv', 'loader': lambda f, z: load_miso_hub(f, 'MICHIGAN.HUB', z)},
    'MISO_ARKANSAS_HUB':  {'file': f'{P}/miso_all_hubs_da_lmp_2022_2025.csv', 'loader': lambda f, z: load_miso_hub(f, 'ARKANSAS.HUB', z)},
    'MISO_LOUISIANA_HUB': {'file': f'{P}/miso_all_hubs_da_lmp_2022_2025.csv', 'loader': lambda f, z: load_miso_hub(f, 'LOUISIANA.HUB', z)},
    
    # ── NYISO zones (GridStatus format) ──────────────────────────────────────
    'NYISO_ZONE_J': {'file': f'{P}/nyiso_zone_j_da_lbmp_2022_2025.csv', 'loader': load_nyiso_zone},
    'NYISO_ZONE_F': {'file': f'{P}/nyiso_zone_f_da_lbmp_2022_2025.csv', 'loader': load_nyiso_zone},
    'NYISO_ZONE_A': {'file': f'{P}/nyiso_zone_a_da_lbmp_2022_2025.csv', 'loader': load_nyiso_zone},
    'NYISO_ZONE_G': {'file': f'{P}/nyiso_zone_g_da_lbmp_2022_2025.csv', 'loader': load_nyiso_zone},
    'NYISO_ZONE_K': {'file': f'{P}/nyiso_zone_k_da_lbmp_2022_2025.csv', 'loader': load_nyiso_zone},
    
    # ── ISO-NE (uncomment when data acquired) ────────────────────────────────
    # 'ISONE_HUB': {'file': f'{P}/isone_hub_da_lmp_2022_2025.csv', 'loader': load_isone_hub},
    
    # ── SPP (uncomment when data acquired) ───────────────────────────────────
    # 'SPP_HUB': {'file': f'{P}/spp_hub_da_lmp_2022_2025.csv', 'loader': load_spp_hub},
}

# Daily sources — separate because different resolution
DAILY_SOURCES = {
    # EIA/ICE data comes as yearly files — we'll concat
    'WECC_MIDC': {
        'files': [
            f'{P}/Palo_Verde_Mid_C_2022.xlsx',
            f'{P}/Palo_Verde_Mid_C_2023.xlsx',
            f'{P}/Palo_Verde_Mid_C_2024.xlsx',
            f'{P}/Palo_Verde_Mid_C_2025.xlsx',
        ],
        'hub_name': 'Mid C',
        'loader': load_eia_daily,
    },
    'WECC_PALO_VERDE': {
        'files': [
            f'{P}/Palo_Verde_Mid_C_2022.xlsx',
            f'{P}/Palo_Verde_Mid_C_2023.xlsx',
            f'{P}/Palo_Verde_Mid_C_2024.xlsx',
            f'{P}/Palo_Verde_Mid_C_2025.xlsx',
        ],
        'hub_name': 'Palo Verde',
        'loader': load_eia_daily,
    },
}

print(f'Hourly sources registered: {len(HOURLY_SOURCES)}')
for zid, info in HOURLY_SOURCES.items():
    exists = os.path.exists(info['file'])
    print(f'  {zid:<25} {"✓" if exists else "✗ MISSING"}')

print(f'\nDaily sources registered: {len(DAILY_SOURCES)}')
for zid, info in DAILY_SOURCES.items():
    all_exist = all(os.path.exists(f) for f in info['files'])
    print(f'  {zid:<25} {"✓" if all_exist else "✗ SOME MISSING"}')

Hourly sources registered: 24
  PJM_COMED                 ✓
  PJM_DOM                   ✓
  PJM_AEP                   ✓
  PJM_BGE                   ✓
  PJM_PECO                  ✓
  PJM_PSEG                  ✓
  PJM_PEPCO                 ✓
  ERCOT_LZ_NORTH            ✓
  ERCOT_LZ_SOUTH            ✓
  ERCOT_LZ_WEST             ✓
  ERCOT_LZ_HOUSTON          ✓
  CAISO_NP15                ✓
  CAISO_SP15                ✓
  MISO_MINN_HUB             ✓
  MISO_INDIANA_HUB          ✓
  MISO_ILLINOIS_HUB         ✓
  MISO_MICHIGAN_HUB         ✓
  MISO_ARKANSAS_HUB         ✓
  MISO_LOUISIANA_HUB        ✓
  NYISO_ZONE_J              ✓
  NYISO_ZONE_F              ✓
  NYISO_ZONE_A              ✓
  NYISO_ZONE_G              ✓
  NYISO_ZONE_K              ✓

Daily sources registered: 2
  WECC_MIDC                 ✓
  WECC_PALO_VERDE           ✓


In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# LOAD ALL HOURLY DATA
# ══════════════════════════════════════════════════════════════════════════════

hourly_frames = []
load_errors = []

print('Loading hourly price data...')
print(f'{"Zone":<25} {"Hours":>8} {"Years":>12} {"Mean $/MWh":>12} {"Max $/MWh":>12}')
print('-' * 72)

for zone_id, info in HOURLY_SOURCES.items():
    try:
        df = info['loader'](info['file'], zone_id)
        df['zone_id'] = zone_id
        
        # Add metadata
        if zone_id in ZONE_META.index:
            df['rto'] = ZONE_META.loc[zone_id, 'rto']
            df['interconnection'] = ZONE_META.loc[zone_id, 'interconnection']
        
        yrs = f"{df.index.year.min()}-{df.index.year.max()}"
        print(f'{zone_id:<25} {len(df):>8,} {yrs:>12} {df.price.mean():>12.1f} {df.price.max():>12,.0f}')
        
        hourly_frames.append(df.reset_index())
        
    except FileNotFoundError:
        print(f'{zone_id:<25} {"FILE NOT FOUND":>8}')
        load_errors.append(zone_id)
    except Exception as e:
        print(f'{zone_id:<25} ERROR: {str(e)[:60]}')
        load_errors.append(zone_id)

# Combine
hourly_all = pd.concat(hourly_frames, ignore_index=True)
hourly_all['datetime'] = pd.to_datetime(hourly_all['datetime'])

print(f'\n{"═" * 72}')
print(f'Total hourly: {len(hourly_all):,} rows across {hourly_all.zone_id.nunique()} zones')
if load_errors:
    print(f'Failed to load: {load_errors}')

Loading hourly price data...
Zone                         Hours        Years   Mean $/MWh    Max $/MWh
------------------------------------------------------------------------
PJM_COMED                   35,060    2022-2025         37.3          497
PJM_DOM                     35,060    2022-2025         54.1          678
PJM_AEP                     35,060    2022-2025         44.5          489
PJM_BGE                     35,060    2022-2025         54.4          670
PJM_PECO                    35,060    2022-2025         38.2          532
PJM_PSEG                    35,060    2022-2025         40.1          535
PJM_PEPCO                   35,060    2022-2025         52.4          660
ERCOT_LZ_NORTH              35,059    2022-2025         45.5        4,213
ERCOT_LZ_SOUTH              35,059    2022-2025         45.0        4,138
ERCOT_LZ_WEST               35,059    2022-2025         52.0        4,214
ERCOT_LZ_HOUSTON            35,059    2022-2025         47.8        4,188
CAISO_NP15

In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# LOAD ALL DAILY DATA (Mid-C, Palo Verde)
# ══════════════════════════════════════════════════════════════════════════════

daily_native_frames = []

print('Loading daily (EIA/ICE) price data...')
print(f'{"Zone":<25} {"Days":>8} {"Years":>12} {"Mean $/MWh":>12} {"Max $/MWh":>12}')
print('-' * 72)

for zone_id, info in DAILY_SOURCES.items():
    try:
        yearly_frames = []
        for fp in info['files']:
            try:
                df_yr = info['loader'](fp, info['hub_name'], zone_id)
                yearly_frames.append(df_yr)
            except FileNotFoundError:
                pass
            except Exception as e:
                print(f'  Warning: {os.path.basename(fp)}: {str(e)[:60]}')
        
        if yearly_frames:
            df = pd.concat(yearly_frames).sort_index()
            df = df[~df.index.duplicated(keep='last')]
            df['zone_id'] = zone_id
            
            if zone_id in ZONE_META.index:
                df['rto'] = ZONE_META.loc[zone_id, 'rto']
                df['interconnection'] = ZONE_META.loc[zone_id, 'interconnection']
            
            yrs = f"{df.index.year.min()}-{df.index.year.max()}"
            print(f'{zone_id:<25} {len(df):>8,} {yrs:>12} {df.price.mean():>12.1f} {df.price.max():>12,.0f}')
            
            daily_native_frames.append(df.reset_index())
        else:
            print(f'{zone_id:<25} NO FILES FOUND')
            
    except Exception as e:
        print(f'{zone_id:<25} ERROR: {str(e)[:60]}')

if daily_native_frames:
    daily_native = pd.concat(daily_native_frames, ignore_index=True)
    daily_native['date'] = pd.to_datetime(daily_native['date'])
    print(f'\nTotal daily native: {len(daily_native):,} rows across {daily_native.zone_id.nunique()} zones')
else:
    daily_native = pd.DataFrame()
    print('\nNo daily data loaded.')

Loading daily (EIA/ICE) price data...
Zone                          Days        Years   Mean $/MWh    Max $/MWh
------------------------------------------------------------------------
WECC_MIDC                      933    2022-2025         72.8        1,001
WECC_PALO_VERDE                930    2022-2025         62.2        1,000

Total daily native: 1,863 rows across 2 zones


In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# BUILD DAILY PEAK DATASET
# ══════════════════════════════════════════════════════════════════════════════
# Collapse hourly → daily on-peak average, then merge with native daily
# This creates an apples-to-apples comparison surface for all zones

print('Collapsing hourly data to daily on-peak averages...')
print(f'Peak window: HB {PEAK_HOURS_START}:00 – {PEAK_HOURS_END}:00')

# Filter to peak hours
hourly_peak = hourly_all[
    (hourly_all['datetime'].dt.hour >= PEAK_HOURS_START) &
    (hourly_all['datetime'].dt.hour <= PEAK_HOURS_END)
].copy()

hourly_peak['date'] = hourly_peak['datetime'].dt.date

# Collapse: daily mean of peak-hour prices per zone
daily_from_hourly = hourly_peak.groupby(['date', 'zone_id']).agg(
    price=('price', 'mean'),
    rto=('rto', 'first'),
    interconnection=('interconnection', 'first'),
    peak_hours_count=('price', 'count'),
).reset_index()

daily_from_hourly['date'] = pd.to_datetime(daily_from_hourly['date'])
daily_from_hourly['data_source'] = 'collapsed_hourly'

print(f'  Collapsed hourly → {len(daily_from_hourly):,} daily rows')

# Tag native daily
if len(daily_native) > 0:
    daily_native_tagged = daily_native.copy()
    daily_native_tagged['data_source'] = 'native_daily'
    daily_native_tagged['peak_hours_count'] = 0  # native daily = already peak-only
    
    # Combine
    daily_all = pd.concat([
        daily_from_hourly[['date', 'zone_id', 'price', 'rto', 'interconnection', 'data_source', 'peak_hours_count']],
        daily_native_tagged[['date', 'zone_id', 'price', 'rto', 'interconnection', 'data_source', 'peak_hours_count']],
    ], ignore_index=True)
else:
    daily_all = daily_from_hourly.copy()

daily_all = daily_all.sort_values(['zone_id', 'date']).reset_index(drop=True)

print(f'\nCombined daily dataset: {len(daily_all):,} rows across {daily_all.zone_id.nunique()} zones')
print(f'\n{"Zone":<25} {"Days":>6} {"Source":<20} {"Mean $/MWh":>12} {"Max $/MWh":>12}')
print('-' * 78)
for (zid, src), grp in daily_all.groupby(['zone_id', 'data_source']):
    print(f'{zid:<25} {len(grp):>6} {src:<20} {grp.price.mean():>12.1f} {grp.price.max():>12,.0f}')

Collapsing hourly data to daily on-peak averages...
Peak window: HB 6:00 – 21:00
  Collapsed hourly → 34,358 daily rows

Combined daily dataset: 36,221 rows across 26 zones

Zone                        Days Source                 Mean $/MWh    Max $/MWh
------------------------------------------------------------------------------
CAISO_NP15                  1106 collapsed_hourly             49.5          503
CAISO_SP15                  1110 collapsed_hourly             40.6          461
ERCOT_LZ_HOUSTON            1461 collapsed_hourly             56.9        1,573
ERCOT_LZ_NORTH              1461 collapsed_hourly             54.0        1,574
ERCOT_LZ_SOUTH              1461 collapsed_hourly             53.1        1,555
ERCOT_LZ_WEST               1461 collapsed_hourly             57.2        1,568
MISO_ARKANSAS_HUB           1461 collapsed_hourly             39.7          189
MISO_ILLINOIS_HUB           1461 collapsed_hourly             44.6          192
MISO_INDIANA_HUB           

In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# SAVE PARQUET FILES
# ══════════════════════════════════════════════════════════════════════════════

hourly_path = os.path.join(OUTPUT_DIR, 'hourly_zone_prices.parquet')
daily_path  = os.path.join(OUTPUT_DIR, 'daily_peak_zone_prices.parquet')
meta_path   = os.path.join(OUTPUT_DIR, 'zone_metadata.parquet')

# Save hourly
hourly_all.to_parquet(hourly_path, index=False)
hourly_size = os.path.getsize(hourly_path) / (1024 * 1024)
print(f'Saved: {hourly_path}')
print(f'  {len(hourly_all):,} rows, {hourly_all.zone_id.nunique()} zones, {hourly_size:.1f} MB')

# Save daily
daily_all.to_parquet(daily_path, index=False)
daily_size = os.path.getsize(daily_path) / (1024 * 1024)
print(f'\nSaved: {daily_path}')
print(f'  {len(daily_all):,} rows, {daily_all.zone_id.nunique()} zones, {daily_size:.1f} MB')

# Save metadata
ZONE_META.to_parquet(meta_path)
print(f'\nSaved: {meta_path}')
print(f'  {len(ZONE_META)} zones')

# Also save CSV versions for easy inspection
hourly_all.to_csv(os.path.join(OUTPUT_DIR, 'hourly_zone_prices.csv'), index=False)
daily_all.to_csv(os.path.join(OUTPUT_DIR, 'daily_peak_zone_prices.csv'), index=False)
ZONE_META.to_csv(os.path.join(OUTPUT_DIR, 'zone_metadata.csv'))
print('\nCSV copies also saved for inspection.')

Saved: C:\Users\dunla\repos\data-center-flexibility-resource-adequacy\data\processed\hourly_zone_prices.parquet
  824,518 rows, 24 zones, 11.4 MB

Saved: C:\Users\dunla\repos\data-center-flexibility-resource-adequacy\data\processed\daily_peak_zone_prices.parquet
  36,221 rows, 26 zones, 0.4 MB

Saved: C:\Users\dunla\repos\data-center-flexibility-resource-adequacy\data\processed\zone_metadata.parquet
  26 zones

CSV copies also saved for inspection.


In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# VALIDATION & SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

print('VALIDATION SUMMARY')
print('=' * 90)

# Coverage check per zone per year
print(f'\nHourly data coverage (hours per year, expected ~8760):')
hourly_all['year'] = hourly_all['datetime'].dt.year
coverage = hourly_all.groupby(['zone_id', 'year']).size().unstack(fill_value=0)
print(coverage.to_string())

# Daily coverage
if len(daily_all) > 0:
    print(f'\nDaily data coverage (days per year, expected ~365):')
    daily_all['year'] = daily_all['date'].dt.year
    dcov = daily_all.groupby(['zone_id', 'year']).size().unstack(fill_value=0)
    print(dcov.to_string())

# Quick sanity: ComEd vs each destination — are they on the same time index?
print(f'\nDatetime alignment check (overlap with PJM_COMED):')
comed_idx = set(hourly_all[hourly_all.zone_id == 'PJM_COMED']['datetime'])
for zid in hourly_all.zone_id.unique():
    if zid == 'PJM_COMED':
        continue
    zone_idx = set(hourly_all[hourly_all.zone_id == zid]['datetime'])
    overlap = len(comed_idx & zone_idx)
    print(f'  {zid:<25} {overlap:>6,} overlapping hours out of {len(zone_idx):>6,}')

# Capacity-weighted destination summary
print(f'\nDestination capacity coverage:')
loaded_zones = set(hourly_all.zone_id.unique())
if len(daily_native) > 0:
    loaded_zones |= set(daily_native.zone_id.unique())
    
destinations = ZONE_META[ZONE_META.migration_role == 'cross_ba_destination']
covered = destinations[destinations.index.isin(loaded_zones)]
uncovered = destinations[~destinations.index.isin(loaded_zones)]

print(f'  Covered:   {len(covered):>3} zones, {covered.dc_capacity_mw.sum():>10,.1f} MW')
print(f'  Uncovered: {len(uncovered):>3} zones, {uncovered.dc_capacity_mw.sum():>10,.1f} MW')
if len(uncovered) > 0:
    for zid, row in uncovered.iterrows():
        print(f'    {zid:<25} {row.dc_capacity_mw:>8,.1f} MW')

total_dest = destinations.dc_capacity_mw.sum()
covered_dest = covered.dc_capacity_mw.sum()
print(f'\n  Coverage: {covered_dest/total_dest:.1%} of destination DC capacity has price data')

print(f'\n{"═" * 90}')
print(f'DONE. Analysis notebooks can now load:')
print(f'  hourly  = pd.read_parquet("{hourly_path}")')
print(f'  daily   = pd.read_parquet("{daily_path}")')
print(f'  meta    = pd.read_parquet("{meta_path}")')

VALIDATION SUMMARY

Hourly data coverage (hours per year, expected ~8760):
year                2022  2023  2024  2025
zone_id                                   
CAISO_NP15          1104  8471  8495  8471
CAISO_SP15          1200  8471  8495  8471
ERCOT_LZ_HOUSTON    8758  8759  8783  8759
ERCOT_LZ_NORTH      8758  8759  8783  8759
ERCOT_LZ_SOUTH      8758  8759  8783  8759
ERCOT_LZ_WEST       8758  8759  8783  8759
MISO_ARKANSAS_HUB   8760  8760  8784  8760
MISO_ILLINOIS_HUB   8760  8760  8784  8760
MISO_INDIANA_HUB    8760  8760  8784  8760
MISO_LOUISIANA_HUB  8760  8760  8784  8760
MISO_MICHIGAN_HUB   8760  8760  8784  8760
MISO_MINN_HUB       8760  8760  8784  8760
NYISO_ZONE_A        8759  8759  8783  8759
NYISO_ZONE_F        8759  8759  8783  8759
NYISO_ZONE_G        8759  8759  8783  8759
NYISO_ZONE_J        8759  8759  8783  8759
NYISO_ZONE_K        8759  8759  8783  8759
PJM_AEP             8759  8759  8783  8759
PJM_BGE             8759  8759  8783  8759
PJM_COMED           87